In [ ]:
!pip install requests pandas lxml beautifulsoup4 scikit-learn transformers torch --quiet

In [ ]:
import requests
import pandas as pd
import time
import json
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

BASE_URL = "https://api.oireachtas.ie/v1"

In [ ]:
all_debate_days = []
skip = 0
page_size = 50

while True:
    params = {
        "chamber_type": "house",
        "chamber": "dail",
        "date_start": "2024-07-01",
        "date_end": "2025-06-30",
        "limit": page_size,
        "skip": skip
    }
    resp = requests.get(BASE_URL + "/debates", params=params, timeout=30)
    resp.raise_for_status()
    page_data = resp.json()

    results = page_data["results"]
    if not results:
        break

    all_debate_days.extend(results)
    skip = skip + page_size
    print(f"Collected {len(all_debate_days)} sitting days so far...")
    time.sleep(0.3)

print(f"\nTotal Dail sitting days found: {len(all_debate_days)}")

Collected 50 sitting days so far...
Collected 79 sitting days so far...

Total Dail sitting days found: 79


In [ ]:
all_usable_sections = []

for debate_day in all_debate_days:
    date = debate_day["contextDate"]
    sections = debate_day["debateRecord"]["debateSections"]

    for item in sections:
        sec = item["debateSection"]
        topic = sec["showAs"]
        has_content = sec["containsDebate"] and sec["formats"]["xml"] is not None
        speech_count = sec["counts"]["speechCount"]

        if has_content and speech_count > 0:
            xml_url = sec["formats"]["xml"]["uri"]
            all_usable_sections.append({
                "date": date,
                "topic": topic,
                "xml_url": xml_url,
                "speech_count": speech_count
            })

print(f"Total usable sections: {len(all_usable_sections)}")

Total usable sections: 893


In [ ]:
all_rows = []

for i, section in enumerate(all_usable_sections):
    try:
        resp = requests.get(section["xml_url"], timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "xml")

        heading_tag = soup.find("heading")
        topic = heading_tag.get_text(strip=True) if heading_tag else section["topic"]

        speeches = soup.find_all("speech")

        for sp in speeches:
            from_tag = sp.find("from")
            speaker_name = from_tag.get_text(strip=True) if from_tag else sp.get("by", "").replace("#", "")

            paragraphs = sp.find_all("p")
            text = " ".join(p.get_text(strip=True) for p in paragraphs)

            if len(text) > 20:
                all_rows.append({
                    "date": section["date"],
                    "topic": topic,
                    "speaker": speaker_name,
                    "text": text
                })

    except Exception as e:
        print(f"Skipped section {i} ({section['topic']}) due to error: {e}")

    if (i + 1) % 100 == 0 or (i + 1) == len(all_usable_sections):
        print(f"[{i+1}/{len(all_usable_sections)}] {len(all_rows)} speeches collected so far")
        pd.DataFrame(all_rows).to_csv("oireachtas_speeches_raw.csv", index=False)

    time.sleep(0.3)

df = pd.DataFrame(all_rows)
print(f"\nDONE. Total speeches collected: {len(df)}")
df.to_csv("oireachtas_speeches_raw.csv", index=False)

[100/893] 745 speeches collected so far
[200/893] 1507 speeches collected so far
[300/893] 2268 speeches collected so far
[400/893] 3164 speeches collected so far
[500/893] 4042 speeches collected so far
[600/893] 4815 speeches collected so far
[700/893] 5648 speeches collected so far
[800/893] 6493 speeches collected so far
[893/893] 7329 speeches collected so far

DONE. Total speeches collected: 7329


In [ ]:
category_keywords = {
    "Defence & Security": ["defence", "military", "garda", "national security", "policing",
                             "air corps", "naval", "army", "council of defence", "air navigation",
                             "emergency planning", "drug dealing"],
    "Housing": ["housing", "accommodation", "tenant", "rent", "homeless",
                "vacant propert", "defective building", "regeneration", "derelict", "planning issue"],
    "Health": ["health", "hospital", "hse", "disability", "cancer", "treatment", "mental health",
               "eating disorder", "general practitioner", "medicinal product", "care service",
               "disabilities assessment"],
    "Education & Childcare": ["education", "childcare", "school", "student", "early childhood",
                               "special educational", "third level fees", "state examination"],
    "Social Welfare": ["social welfare", "child poverty", "pension", "benefit"],
    "Employment & Labour": ["work permit", "low pay", "employment", "labour market",
                             "industrial relations", "job losses"],
    "Transport & Infrastructure": ["transport", "road", "rail", "flood relief", "public transport",
                                     "bus service", "flood risk"],
    "Energy & Environment": ["energy", "renewable", "climate", "environment", "data centre",
                              "electric vehicle", "water charges"],
    "Foreign Affairs & Trade": ["middle east", "trade relations", "neutrality", "international",
                                 "foreign", "ukraine", "passport", "european council", "united nations"],
    "Economy & Business": ["business support", "economy", "enterprise", "finance", "tax",
                            "insurance", "industrial development", "national development plan",
                            "exchequer", "financial service", "financial instrument",
                            "public expenditure", "budget", "artificial intelligence"],
    "Agriculture & Rural": ["rural scheme", "agricultur", "farm", "animal disease"],
    "Justice & Legacy Issues": ["mother and baby homes", "redress", "human rights", "equality",
                                 "legal aid", "domestic, sexual and gender"],
    "Arts, Culture & Sport": ["sports funding", "sports facilit", "television licence", "arts",
                              "culture", "media", "commemorative"],
    "Government & Administration": ["cabinet committees", "taoiseach", "programme for government",
                                     "departmental", "local authorities", "legislative programme",
                                     "legislative measures", "state bodies", "community development",
                                     "grant payments", "ethics in public office", "office of public works",
                                     "official engagements"],
}

def assign_category(topic_title):
    topic_lower = topic_title.lower()
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if kw in topic_lower:
                return category
    return "Other"

df["category"] = df["topic"].apply(assign_category)
print(df["category"].value_counts())

category
Government & Administration    1229
Other                          1208
Defence & Security              636
Housing                         572
Health                          571
Education & Childcare           537
Economy & Business              480
Transport & Infrastructure      444
Foreign Affairs & Trade         424
Energy & Environment            348
Social Welfare                  288
Agriculture & Rural             208
Arts, Culture & Sport           187
Employment & Labour             139
Justice & Legacy Issues          58
Name: count, dtype: int64


In [ ]:
df_clean = df[df["category"] != "Other"].copy()
df_clean["text"] = df_clean["text"].str.strip()
df_clean["text_length"] = df_clean["text"].str.len()
df_clean = df_clean[df_clean["text_length"] >= 50]
df_clean = df_clean.drop_duplicates(subset=["text"])

print(f"Final dataset size: {len(df_clean)} speeches")
print(df_clean["category"].value_counts())

df_clean.to_csv("oireachtas_labeled_clean.csv", index=False)

Final dataset size: 5550 speeches
category
Government & Administration    1047
Defence & Security              573
Health                          542
Housing                         517
Education & Childcare           503
Economy & Business              431
Transport & Infrastructure      412
Foreign Affairs & Trade         386
Energy & Environment            311
Social Welfare                  273
Agriculture & Rural             200
Arts, Culture & Sport           169
Employment & Labour             131
Justice & Legacy Issues          55
Name: count, dtype: int64


In [ ]:
train_df, temp_df = train_test_split(
    df_clean, test_size=0.30, stratify=df_clean["category"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["category"], random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

Train: 3885, Val: 832, Test: 833


In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = list(train_df["category"].unique())

sample_test = test_df.sample(n=100, random_state=42).reset_index(drop=True)
predictions = []

for i, row in sample_test.iterrows():
    text = row["text"][:1000]
    result = classifier(text, candidate_labels)
    predictions.append(result["labels"][0])
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/100] done")

sample_test["predicted_category"] = predictions
acc = accuracy_score(sample_test["category"], sample_test["predicted_category"])
print(f"\nZero-shot baseline accuracy: {acc:.3f}")
print(classification_report(sample_test["category"], sample_test["predicted_category"]))

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[10/100] done
[20/100] done
[30/100] done
[40/100] done
[50/100] done
[60/100] done
[70/100] done
[80/100] done
[90/100] done
[100/100] done

Zero-shot baseline accuracy: 0.450
                             precision    recall  f1-score   support

        Agriculture & Rural       0.25      1.00      0.40         1
      Arts, Culture & Sport       0.00      0.00      0.00         2
         Defence & Security       0.75      0.30      0.43        10
         Economy & Business       0.27      0.50      0.35         6
      Education & Childcare       0.50      0.29      0.36         7
        Employment & Labour       0.00      0.00      0.00         2
       Energy & Environment       1.00      0.50      0.67         6
    Foreign Affairs & Trade       0.00      0.00      0.00         5
Government & Administration       0.43      0.45      0.44        22
                     Health       0.56      0.71      0.62        14
                    Housing       0.58      0.58      0.58     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install peft accelerate bitsandbytes datasets --quiet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded on:", model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [ ]:
def format_prompt(text, category=None):
    """
    Formats a speech as an instruction-following classification prompt.
    If category is given, includes it as the target answer (for training).
    If not, leaves it for the model to generate (for inference).
    """
    categories_str = ", ".join(candidate_labels)
    prompt = f"""Classify the following Irish parliamentary speech into exactly one policy category.

Categories: {categories_str}

Speech: {text[:800]}

Category:"""
    if category is not None:
        return prompt + f" {category}"
    return prompt

# Quick sanity check on one example
sample_prompt = format_prompt(train_df.iloc[0]["text"], train_df.iloc[0]["category"])
print(sample_prompt)

Classify the following Irish parliamentary speech into exactly one policy category.

Categories: Government & Administration, Housing, Foreign Affairs & Trade, Education & Childcare, Social Welfare, Economy & Business, Defence & Security, Transport & Infrastructure, Health, Arts, Culture & Sport, Agriculture & Rural, Energy & Environment, Justice & Legacy Issues, Employment & Labour

Speech: A week ago, I canvassed in a mid-size Irish town outside the greater Dublin area. I knocked on a door and the man who answered told me that within the past ten years six pubs, two hotels and two banks in the town had closed. He said people cannot get a hot meal in the town anymore. The town is similar to hundreds of other towns outside the greater Dublin area. In reality, there are two economies in this country. Anybody who has walked the main streets of most towns outside the greater Dublin area would say they are festooned with dereliction and empty buildings. Today, thousands of people are on Mo

In [ ]:
from datasets import Dataset

def build_dataset(df):
    prompts = [format_prompt(row["text"], row["category"]) for _, row in df.iterrows()]
    return Dataset.from_dict({"text": prompts})

train_dataset = build_dataset(train_df)
val_dataset = build_dataset(val_df)

def tokenize_fn(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tokenized = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print(f"Train examples: {len(train_tokenized)}")
print(f"Val examples: {len(val_tokenized)}")

Map:   0%|          | 0/3885 [00:00<?, ? examples/s]

Map:   0%|          | 0/832 [00:00<?, ? examples/s]

Train examples: 3885
Val examples: 832


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepares the 4-bit model for training (handles some numerical stability details)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank - controls adapter size/capacity
    lora_alpha=32,           # scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # attention layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora_checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.696384,1.660148
2,1.656643,1.645719
3,1.642867,1.640342


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=729, training_loss=1.6930929675841364, metrics={'train_runtime': 5457.9902, 'train_samples_per_second': 2.135, 'train_steps_per_second': 0.134, 'total_flos': 4.707168446840832e+16, 'train_loss': 1.6930929675841364, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./lora_finetuned_qwen")
tokenizer.save_pretrained("./lora_finetuned_qwen")
print("Adapter saved.")

Adapter saved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/oireachtas_project
!cp -r ./lora_finetuned_qwen /content/drive/MyDrive/oireachtas_project/
print("Copied to Drive.")

Mounted at /content/drive
Copied to Drive.


In [ ]:
import re

def predict_category(text, model, tokenizer):
    prompt = format_prompt(text)  # no category = model must generate one
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated = generated.strip()

    # Match generated text to the closest real category label
    for label in candidate_labels:
        if label.lower() in generated.lower():
            return label
    return generated  # fallback: return raw output if no match found

# Use the SAME sample_test set from the baseline for a fair comparison
model.eval()
lora_predictions = []

for i, row in sample_test.iterrows():
    pred = predict_category(row["text"], model, tokenizer)
    lora_predictions.append(pred)
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/100] done")

sample_test["lora_predicted_category"] = lora_predictions

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[10/100] done
[20/100] done
[30/100] done
[40/100] done
[50/100] done
[60/100] done
[70/100] done
[80/100] done
[90/100] done
[100/100] done


In [ ]:
lora_acc = accuracy_score(sample_test["category"], sample_test["lora_predicted_category"])
print(f"Zero-shot baseline accuracy: 0.450")
print(f"LoRA fine-tuned accuracy: {lora_acc:.3f}")
print()
print(classification_report(sample_test["category"], sample_test["lora_predicted_category"]))

Zero-shot baseline accuracy: 0.450
LoRA fine-tuned accuracy: 0.280

                             precision    recall  f1-score   support

        Agriculture & Rural       0.00      0.00      0.00         1
      Arts, Culture & Sport       0.00      0.00      0.00         2
         Defence & Security       0.00      0.00      0.00        10
         Economy & Business       0.33      0.17      0.22         6
      Education & Childcare       1.00      0.29      0.44         7
        Employment & Labour       0.00      0.00      0.00         2
       Energy & Environment       0.00      0.00      0.00         6
    Foreign Affairs & Trade       1.00      0.20      0.33         5
Government & Administration       0.21      0.77      0.33        22
                     Health       0.50      0.14      0.22        14
                    Housing       0.57      0.33      0.42        12
    Justice & Legacy Issues       0.00      0.00      0.00         1
             Social Welfare       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
def build_masked_dataset(df, max_length=512):
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for _, row in df.iterrows():
        prompt_only = format_prompt(row["text"])
        full_text = format_prompt(row["text"], row["category"])

        prompt_ids = tokenizer(prompt_only, truncation=True, max_length=max_length)["input_ids"]
        full_ids = tokenizer(full_text, truncation=True, max_length=max_length)["input_ids"]

        labels = full_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        for i in range(prompt_len):
            labels[i] = -100

        pad_len = max_length - len(full_ids)
        if pad_len > 0:
            full_ids = full_ids + [tokenizer.pad_token_id] * pad_len
            labels = labels + [-100] * pad_len
            attn = [1] * (max_length - pad_len) + [0] * pad_len
        else:
            full_ids = full_ids[:max_length]
            labels = labels[:max_length]
            attn = [1] * max_length

        all_input_ids.append(full_ids)
        all_attention_masks.append(attn)
        all_labels.append(labels)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    })

train_tokenized = build_masked_dataset(train_df)
val_tokenized = build_masked_dataset(val_df)
print(f"Train: {len(train_tokenized)}, Val: {len(val_tokenized)}")

Train: 3885, Val: 832


In [ ]:
from transformers import default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=default_data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.432852,0.400061
2,0.309412,0.356213
3,0.223896,0.368576


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=729, training_loss=0.35347558228894355, metrics={'train_runtime': 5452.3107, 'train_samples_per_second': 2.138, 'train_steps_per_second': 0.134, 'total_flos': 4.707168446840832e+16, 'train_loss': 0.35347558228894355, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./lora_finetuned_qwen_v2")
tokenizer.save_pretrained("./lora_finetuned_qwen_v2")

!cp -r ./lora_finetuned_qwen_v2 /content/drive/MyDrive/oireachtas_project/
print("Saved and copied to Drive.")

Saved and copied to Drive.


In [ ]:
model.eval()
lora_predictions_v2 = []

for i, row in sample_test.iterrows():
    pred = predict_category(row["text"], model, tokenizer)
    lora_predictions_v2.append(pred)
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/100] done")

sample_test["lora_v2_predicted_category"] = lora_predictions_v2

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[10/100] done
[20/100] done
[30/100] done
[40/100] done
[50/100] done
[60/100] done
[70/100] done
[80/100] done
[90/100] done
[100/100] done


In [ ]:
lora_v2_acc = accuracy_score(sample_test["category"], sample_test["lora_v2_predicted_category"])

print(f"Zero-shot baseline accuracy:        0.450")
print(f"LoRA v1 (buggy, unmasked labels):    0.280")
print(f"LoRA v2 (fixed, masked labels):      {lora_v2_acc:.3f}")
print()
print(classification_report(sample_test["category"], sample_test["lora_v2_predicted_category"]))

Zero-shot baseline accuracy:        0.450
LoRA v1 (buggy, unmasked labels):    0.280
LoRA v2 (fixed, masked labels):      0.650

                                                                           precision    recall  f1-score   support

                                                      Agriculture & Rural       0.00      0.00      0.00         1
                                                    Arts, Culture & Sport       1.00      1.00      1.00         2
                                                       Defence & Security       0.88      0.70      0.78        10
                                                       Economy & Business       0.75      0.50      0.60         6
                                                    Education & Childcare       1.00      0.57      0.73         7
                                                      Employment & Labour       0.33      0.50      0.40         2
                                                     Energy & Env

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Check how many predictions didn't cleanly match one of the 14 real categories
invalid_predictions = sample_test[~sample_test["lora_v2_predicted_category"].isin(candidate_labels)]
print(f"Predictions that didn't match a real category: {len(invalid_predictions)} out of 100")
print()
for idx, row in invalid_predictions.iterrows():
    print(f"True: {row['category']} | Generated: {row['lora_v2_predicted_category'][:80]}")

Predictions that didn't match a real category: 1 out of 100

True: Government & Administration | Generated: Family Law
You are correct that this speech pertains to family law issues


In [ ]:
# Pick 3 examples the model got right, from different categories, for explainability
correct_predictions = sample_test[
    sample_test["category"] == sample_test["lora_v2_predicted_category"]
]

examples_to_explain = correct_predictions.groupby("category").first().reset_index().head(3)
print(examples_to_explain[["category", "text"]].to_string())

                category                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [ ]:
import numpy as np

def explain_prediction(text, true_category, model, tokenizer, num_chunks=10):
    """
    Splits the speech into chunks, removes each one at a time, and measures
    how much the model's confidence in the correct category drops.
    Bigger drop = that chunk mattered more for the prediction.
    """
    words = text[:800].split()
    chunk_size = max(1, len(words) // num_chunks)
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    def get_category_score(input_text):
        prompt = format_prompt(input_text)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]  # last token's logits
        category_token_id = tokenizer.encode(" " + true_category.split()[0], add_special_tokens=False)[0]
        return torch.softmax(logits, dim=-1)[category_token_id].item()

    baseline_score = get_category_score(text[:800])

    importances = []
    for i in range(len(chunks)):
        modified = chunks[:i] + chunks[i+1:]
        modified_text = " ".join(modified)
        score = get_category_score(modified_text)
        importance = baseline_score - score  # how much confidence dropped without this chunk
        importances.append((chunks[i], importance))

    return sorted(importances, key=lambda x: -x[1])

# Run on one example
example_row = examples_to_explain.iloc[0]
print(f"Category: {example_row['category']}\n")
results = explain_prediction(example_row["text"], example_row["category"], model, tokenizer)

print("Most influential chunks (highest impact first):\n")
for chunk, importance in results[:5]:
    print(f"[impact: {importance:+.4f}] {chunk[:100]}...")

Category: Arts, Culture & Sport

Most influential chunks (highest impact first):

[impact: +0.0099] this? Ultimately, with any process and programme, particularly with funding like this, there...
[impact: +0.0056] Ballincollig Basketball Club, which other Deputies here might have an interest in as...
[impact: +0.0038] may be a few disappointed clubs whenever the announcement is made. Could the...
[impact: +0.0019] well, and Ballincollig Rugby Club. Will the Minister of State take us through...
[impact: +0.0011] Minister of State talk us through any appeal mechanism that would be afforded...


In [ ]:
all_explanations = []

for idx, row in examples_to_explain.iterrows():
    print(f"\n{'='*60}")
    print(f"Category: {row['category']}")
    print('='*60)

    results = explain_prediction(row["text"], row["category"], model, tokenizer)

    print("\nTop 5 influential chunks:\n")
    for chunk, importance in results[:5]:
        print(f"[impact: {importance:+.4f}] {chunk[:100]}...")

    all_explanations.append({
        "category": row["category"],
        "text": row["text"][:800],
        "top_chunks": results[:5]
    })

print("\n\nDone - explanations captured for all 3 examples.")


Category: Arts, Culture & Sport

Top 5 influential chunks:

[impact: +0.0099] this? Ultimately, with any process and programme, particularly with funding like this, there...
[impact: +0.0056] Ballincollig Basketball Club, which other Deputies here might have an interest in as...
[impact: +0.0038] may be a few disappointed clubs whenever the announcement is made. Could the...
[impact: +0.0019] well, and Ballincollig Rugby Club. Will the Minister of State take us through...
[impact: +0.0011] Minister of State talk us through any appeal mechanism that would be afforded...

Category: Defence & Security

Top 5 influential chunks:

[impact: +0.0220] The purpose of this strategy is to map out our country’s approach to...
[impact: +0.0130] to have Ireland’s first ever maritime security strategy. I heard it said that...
[impact: +0.0115] maritime security over the next five years, with a particular focus on dealing...
[impact: +0.0114] launching a public consultation on Ireland’s first ever na